# Đồ án môn CS419 – Truy xuất Thông tin
## Tập dữ liệu Cranfield

Notebook này trình bày toàn bộ pipeline truy xuất thông tin trên tập dữ liệu **Cranfield** (1 400 tài liệu, 225 câu truy vấn) sử dụng hai mô hình:
- **Vector Space Model (VSM)** với trọng số TF-IDF và độ đo Cosine Similarity
- **Okapi BM25**

In [2]:
import re, math, os, csv
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from num2words import num2words

for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg, quiet=True)

CRANFIELD = Path('Cranfield')
TEST      = Path('TEST')


## 1. Tải Dữ liệu (Data Loading)

Tập Cranfield gồm:
- **1 400** tài liệu khoa học về khí động học
- **225** câu truy vấn tiếng Anh
- **225** file đánh giá độ liên quan (relevance judgements)

In [3]:
def load_documents(folder):
    docs = {}
    for f in sorted(os.listdir(folder)):
        if f.endswith('.txt'):
            doc_id = int(f.split('.')[0])
            docs[doc_id] = open(os.path.join(folder, f), encoding='utf-8').read()
    return docs

def load_queries(query_file):
    queries = {}
    with open(query_file, encoding='utf-8') as f:
        for row in csv.reader(f, delimiter='\t'):
            if len(row) >= 2:
                queries[int(row[0])] = row[1]
    return queries

def load_relevance(result_path):
    rels = defaultdict(list)
    for fname in os.listdir(result_path):
        qid = int(fname.split('.')[0])
        df  = pd.read_csv(os.path.join(result_path, fname),
                          sep=r'\s+', header=None,
                          names=['QueryID','DocID','Rating'], engine='python')
        df  = df.dropna(subset=['DocID','Rating'])
        rels[qid] = [int(r.DocID) for r in df.itertuples() if int(r.Rating) != -1]
    return rels


In [4]:
documents  = load_documents(CRANFIELD)
queries    = load_queries(TEST / 'query.txt')
query_rels = load_relevance(TEST / 'RES')

print(f"Tài liệu : {len(documents)}")
print(f"Câu truy vấn : {len(queries)}")
print(f"Relevance files : {len(query_rels)}")


Tài liệu : 1400
Câu truy vấn : 225
Relevance files : 225


## 2. Tiền xử lý (Preprocessing)

Pipeline xử lý văn bản gồm 5 bước:

```
Văn bản thô
  → Lowercase
  → Mở rộng viết tắt (Regex)
  → Chuyển số thành chữ
  → Tokenize + Lọc Stopwords
  → Stemming (Snowball)
```

### 2.1. Làm sạch & Mở rộng viết tắt

In [5]:
ABBREVIATIONS = {
    r'\bfig\.?\b':    'figure',
    r'\bref\.?\b':    'reference',
    r'\bapprox\.?\b': 'approximately',
    r'\beq\.?\b':     'equation',
    r'\bsq\.?\b':     'square',
    r'\bno\.?\b':     'number',
    r'\be\.g\.?\b':  'for example',
    r'\bi\.e\.?\b':  'that is',
    r'\bsec\.?\b':    'section',
}

CUSTOM_STOPWORDS = {'ii','iii','iv','vi','vii','viii','ix','xi','xii'}

def replace_number(match):
    try:
        return num2words(float(match.group())).replace('-',' ').replace(',','')
    except Exception:
        return match.group()


### 2.2. Tokenization & Stopwords

In [6]:
def build_stopwords():
    sw = set(stopwords.words('english'))
    sw.update(CUSTOM_STOPWORDS)
    return sw

STOP_WORDS = build_stopwords()


#### Ví dụ chạy tay Pipeline Tiền Xử Lý (Query 1)

**Câu gốc:**
`"what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft ."`

**Bước 1: Chuyển chữ thường & Loại bỏ ký tự đặc biệt (Regex)**
`"what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft"`

**Bước 2: Tách từ (Tokenize)**
`['what', 'similarity', 'laws', 'must', 'be', 'obeyed', 'when', 'constructing', 'aeroelastic', 'models', 'of', 'heated', 'high', 'speed', 'aircraft']`

**Bước 3: Lọc Stopword & Từ ngắn (length <= 2)**
* Các từ bị loại: `what`, `must`, `be`, `when`, `of` (Stopword)
* Các từ giữ lại: `['similarity', 'laws', 'obeyed', 'constructing', 'aeroelastic', 'models', 'heated', 'high', 'speed', 'aircraft']`

**Bước 4: Stemming (Snowball)**
* `similarity` → `similar`
* `laws` → `law`
* `obeyed` → `obey`
* `constructing` → `construct`
* `aeroelastic` → `aeroelast`
* `models` → `model`
* `heated` → `heat`
* `high` → `high`
* `speed` → `speed`
* `aircraft` → `aircraft`

**Kết quả cuối cùng:**
`['similar', 'law', 'obey', 'construct', 'aeroelast', 'model', 'heat', 'high', 'speed', 'aircraft']`


### 2.3. Stemming

#### Thuật toán Snowball Stemmer

Snowball Stemmer áp dụng tập quy tắc **suffix-stripping** để đưa từ về dạng gốc. Ví dụ:

| Từ gốc | Sau Stemming |
|--------|-------------|
| *flowing* | flow |
| *studies* | studi |
| *aerodynamics* | aerodynam |
| *boundary* | boundari |

Mục đích: đồng nhất biến thể từ để tăng khả năng match giữa query và tài liệu.

In [7]:
STEMMER = SnowballStemmer('english')

examples = ['flowing', 'studies', 'aerodynamics', 'boundary', 'computed', 'material', 'photoelastic']
for w in examples:
    print(f"  {w:20s} → {STEMMER.stem(w)}")


  flowing              → flow
  studies              → studi
  aerodynamics         → aerodynam
  boundary             → boundari
  computed             → comput
  material             → materi
  photoelastic         → photoelast


### 2.4. Pipeline hoàn chỉnh

In [8]:
def process_document(document, stop_words=None, stemmer=None):
    if stop_words is None: stop_words = STOP_WORDS
    if stemmer    is None: stemmer    = STEMMER

    text = document.lower().replace('-', ' ')
    for pat, repl in ABBREVIATIONS.items():
        text = re.sub(pat, repl, text)

    text = re.sub(r'\b\d+(?:\.\d+)?\b', replace_number, text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens
              if not (len(w) <= 2 and w.isalpha()) and w not in stop_words]
    return [stemmer.stem(t) for t in tokens]

def process_documents(doc_ids, documents):
    processed, all_tokens = {}, []
    for doc_id in doc_ids:
        tokens = process_document(documents[doc_id])
        processed[doc_id] = tokens
        all_tokens.extend(tokens)
    return processed, all_tokens


In [9]:
doc_ids = list(documents.keys())
processed_docs, all_tokens = process_documents(doc_ids, documents)


In [10]:
# Thống kê trước xử lý (áp dụng regex cơ bản để đếm token sạch)
def raw_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

raw_tokens_all = []
for doc in documents.values():
    raw_tokens_all.extend(raw_tokenize(doc))

raw_vocab   = set(raw_tokens_all)
raw_avg_len = len(raw_tokens_all) / len(documents)

# Thống kê sau xử lý
proc_vocab   = set(all_tokens)
proc_avg_len = sum(len(t) for t in processed_docs.values()) / len(processed_docs)

before    = [len(raw_vocab),  round(raw_avg_len,  2)]
after     = [len(proc_vocab), round(proc_avg_len, 2)]
reduction = [f"{(b-a)/b*100:.2f}%" for b, a in zip(before, after)]

stats = pd.DataFrame({
    'Trước xử lý': before,
    'Sau xử lý'  : after,
    'Giảm'       : reduction,
}, index=['Term', 'Độ dài trung bình'])

stats.style\
    .set_caption("Thống kê Tiền xử lý")\
    .set_properties(**{'text-align': 'right'})\
    .set_properties(subset=pd.IndexSlice[:, ['Trước xử lý','Sau xử lý']], **{'font-weight': 'bold'})\
    .set_table_styles([
        {'selector': 'th', 'props': [('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
        {'selector': 'td', 'props': [('padding','8px 16px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f4f5ff')]},
        {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px')]},
    ])


,Trước xử lý,Sau xử lý,Giảm
Term,7472.000000,4452.000000,40.42%
Độ dài trung bình,161.910000,95.580000,40.97%


## 3. Xây dựng Từ điển & Chỉ mục

### 3.1. Vocabulary

In [11]:
def build_vocabulary(processed_docs):
    all_terms = set()
    for tokens in processed_docs.values():
        all_terms.update(tokens)
    vocab      = sorted(all_terms)
    vocab_index = {term: i for i, term in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 4452


### 3.2. Inverted Index

#### Công thức Inverted Index

Cấu trúc lưu trữ:
```
term → {
  'nDoc'    : số tài liệu chứa term,
  'postings': [(doc_id, tfidf_weight), ...]
}
```
Giúp tra cứu nhanh O(1) thay vì quét toàn bộ ma trận.

In [12]:
def compute_tf(tokens, vocab_index):
    tf = np.zeros(len(vocab_index))
    for t in tokens:
        if t in vocab_index:
            tf[vocab_index[t]] += 1
    n = len(tokens)
    return tf / n if n > 0 else tf

def compute_global_df(processed_docs, vocab_index):
    df = np.zeros(len(vocab_index))
    for tokens in processed_docs.values():
        for t in set(tokens):
            if t in vocab_index:
                df[vocab_index[t]] += 1
    return df


## 4. Các Mô hình Truy xuất

### 4.1. Vector Space Model (VSM)

Mỗi tài liệu và câu truy vấn được biểu diễn dưới dạng **vector trọng số** trong không gian từ vựng. Tài liệu có vector cosine gần nhất với query sẽ được xếp hạng cao nhất.

#### Công thức TF

$$TF(t,d) = \frac{f_{t,d}}{|d|}$$

Trong đó $f_{t,d}$ là số lần term $t$ xuất hiện trong tài liệu $d$, $|d|$ là **tổng số token** trong $d$ (chuẩn hóa theo độ dài).

**Ví dụ:** doc có 95 token, term *'flow'* xuất hiện 3 lần → $TF = 3/95 \approx 0.0316$

#### Công thức IDF (VSM)

$$IDF(t) = \ln\left(\frac{N}{df_t}\right)$$

Trong đó $N = 1400$ (tổng số tài liệu), $df_t$ là số tài liệu chứa term $t$. Dùng logarithm tự nhiên $\ln$ (hàm `math.log` trong Python).

#### Công thức TF-IDF

$$w(t,d) = TF(t,d) \times IDF(t) = \frac{f_{t,d}}{|d|} \times \ln\frac{N}{df_t}$$

In [13]:
def compute_tfidf(processed_docs, vocab_index):
    N = len(processed_docs)
    global_df = compute_global_df(processed_docs, vocab_index)
    sorted_vocab = sorted(vocab_index, key=vocab_index.get)
    tfidf = np.zeros((N, len(vocab_index)))

    for doc_id, tokens in processed_docs.items():
        tf = compute_tf(tokens, vocab_index)
        for i in range(len(sorted_vocab)):
            if tf[i] > 0 and global_df[i] > 0:
                tfidf[doc_id - 1][i] = round(tf[i] * math.log(N / global_df[i]), 6)
    return tfidf

def cosine_similarity(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return np.dot(v1, v2) / (n1 * n2) if n1 and n2 else 0.0

def create_vsm_index(tfidf, processed_docs, vocab_index):
    df_dict = defaultdict(set)
    for doc_id, tokens in processed_docs.items():
        for t in set(tokens): df_dict[t].add(doc_id)

    index = {}
    for term, doc_ids in df_dict.items():
        idx = vocab_index.get(term)
        if idx is None: continue
        postings = [(d, tfidf[d-1][idx]) for d in doc_ids if tfidf[d-1][idx] > 0]
        index[term] = {'nDoc': len(doc_ids), 'postings': postings}
    return index

def search_vsm(query, tfidf, vsm_index, vocab_index, k=20):
    q_tokens = process_document(query)
    q_vec    = np.zeros(len(vocab_index))
    for t in q_tokens:
        if t in vocab_index: q_vec[vocab_index[t]] += 1
    n = len(q_tokens)
    if n > 0: q_vec /= n

    N = tfidf.shape[0]
    for t in set(q_tokens):
        if t in vsm_index:
            idx = vocab_index[t]
            q_vec[idx] *= math.log(N / vsm_index[t]['nDoc'])

    candidates = set()
    for t in q_tokens:
        if t in vsm_index:
            candidates.update(d for d, _ in vsm_index[t]['postings'])

    sims = [(d, cosine_similarity(q_vec, tfidf[d-1])) for d in candidates]
    return sorted(sims, key=lambda x: x[1], reverse=True)[:k]


In [14]:
tfidf_matrix = compute_tfidf(processed_docs, vocab_index)
vsm_index    = create_vsm_index(tfidf_matrix, processed_docs, vocab_index)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


TF-IDF matrix shape: (1400, 4452)


#### Triển khai VSM

In [15]:
# Dùng Query 1 theo yêu cầu để minh họa
demo_query = queries[1]
print(f"Query 1: {demo_query}")
print("\nTop 5 kết quả (VSM):")
for doc_id, score in search_vsm(demo_query, tfidf_matrix, vsm_index, vocab_index, k=5):
    print(f"  Doc {doc_id:4d}  score={score:.4f}")


Query 1: what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .

Top 5 kết quả (VSM):
  Doc   51  score=0.2780
  Doc  184  score=0.2437
  Doc   12  score=0.2185
  Doc  359  score=0.1956
  Doc  746  score=0.1916


#### Ví dụ tính tay TF-IDF (1 term cụ thể)

**Ví dụ: "aeroelast" trong tài liệu số 51**
+ N: tổng số tài liệu (=1400)
+ \|d\|: độ dài tài liệu số 51 (=162)
+ DF: số tài liệu chứa từ "aeroelast" (=113)
+ TF(f): số lần "aeroelast" xuất hiện trong tài liệu 51 (=5)

| TERM | TF(f) | \|d\| | DF | IDF | W |
|:---:|:---:|:---:|:---:|:---:|:---:|
| aeroelast | 5 | 162 | 113 | 2.517 | 0.0777 |

$$idf_{aeroelast} = \ln\left(\frac{1400}{113}\right) \approx 2.517$$

$$w_{aeroelast} = \frac{5}{162} \times 2.517 \approx 0.0777$$


### 4.2. Okapi BM25

BM25 là mô hình xác suất, bổ sung **chuẩn hóa độ dài tài liệu** ($b$) và **bão hòa tần suất** ($k_1$) so với TF-IDF thông thường.

#### Công thức IDF (BM25)

$$IDF_{BM25}(t) = \ln\left(\frac{N - df_t + 0.5}{df_t + 0.5} + 1\right)$$

Khác với VSM: BM25 dùng công thức IDF có tham số smoothing $+1$ bên trong, tránh IDF âm khi $df_t > N/2$.

#### Công thức Score BM25

$$\text{score}(d,q) = \sum_{t \in q} IDF_{BM25}(t) \cdot \frac{f_{t,d} \cdot (k_1+1)}{f_{t,d} + k_1\left(1 - b + b \cdot \dfrac{|d|}{avgdl}\right)}$$

Tham số tối ưu (qua Grid Search): $k_1 = 2.0$, $b = 0.6$, $avgdl \approx 91$ token/tài liệu

In [16]:
class BM25:
    def __init__(self, processed_docs, k1=2.0, b=0.6):
        self.k1, self.b = k1, b
        self.N   = len(processed_docs)
        self.doc_len   = {}
        self.doc_freqs = {}
        self.nd        = {}
        self.idf       = {}
        self.avgdl     = 0
        self._initialize(processed_docs)

    def _initialize(self, docs):
        total = 0
        for doc_id, tokens in docs.items():
            self.doc_len[doc_id] = len(tokens)
            total += len(tokens)
            freq = Counter(tokens)
            self.doc_freqs[doc_id] = freq
            for w in freq: self.nd[w] = self.nd.get(w, 0) + 1

        self.avgdl = total / self.N if self.N > 0 else 0
        for w, df in self.nd.items():
            self.idf[w] = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def get_scores(self, query_tokens, k=20):
        scores = {}
        for doc_id, freq in self.doc_freqs.items():
            s, dl = 0.0, self.doc_len[doc_id]
            for t in query_tokens:
                if t not in freq: continue
                f = freq[t]
                num = f * (self.k1 + 1)
                den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                s  += self.idf.get(t, 0) * (num / den)
            if s > 0: scores[doc_id] = s
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

def search_bm25(query, bm25_model, k=20):
    tokens = process_document(query)
    return bm25_model.get_scores(tokens, k=k)


In [17]:
bm25_model = BM25(processed_docs)
print(f"avgdl = {bm25_model.avgdl:.1f} tokens  |  N = {bm25_model.N}")


avgdl = 95.6 tokens  |  N = 1400


#### Demo BM25 – Query 1

In [18]:
print(f"Query 1: '{queries[1]}'")
print("\nTop 5 kết quả (BM25):")
for doc_id, score in search_bm25(queries[1], bm25_model, k=5):
    print(f"  Doc {doc_id:4d}  BM25 = {score:.4f}")


Query 1: 'what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .'

Top 5 kết quả (BM25):
  Doc   51  BM25 = 25.3230
  Doc  486  BM25 = 22.3925
  Doc   12  BM25 = 20.3612
  Doc  184  BM25 = 19.1581
  Doc  878  BM25 = 17.3155


#### Ví dụ tính tay BM25 (1 term cụ thể)

**Ví dụ: "aeroelast" trong tài liệu số 51 (BM25)**
+ N: tổng số tài liệu (=1400)
+ DF: số tài liệu chứa từ "aeroelast" (=113)
+ TF(f): số lần "aeroelast" xuất hiện trong tài liệu 51 (=5)
+ k1: tham số bão hòa tần suất (=2.0)
+ b: tham số chuẩn hóa độ dài (=0.6)
+ \|d\|: độ dài tài liệu 51 (=162)
+ avgdl: độ dài trung bình của tài liệu (=91.22)

| TERM | TF(f) | DF | IDF | TF_comp | SCORE |
|:---:|:---:|:---:|:---:|:---:|:---:|
| aeroelast | 5 | 113 | 2.515 | 1.831 | 4.606 |

$$idf_{aeroelast} = \ln\left(\frac{1400 - 113 + 0.5}{113 + 0.5} + 1\right) \approx 2.515$$

$$score_{aeroelast} = 2.515 \times \frac{5 \times (2.0 + 1)}{5 + 2.0 \times \left(1 - 0.6 + 0.6 \times \frac{162}{91.22}\right)} = 2.515 \times \frac{15.000}{8.192} \approx 4.606$$


## 5. Đánh giá (Evaluation)

### Công thức MAP, P@k, Recall@k

$$MAP = \frac{1}{|Q|} \sum_{q=1}^{|Q|} AP(q) \qquad AP(q) = \frac{1}{R_q} \sum_{k=1}^{n} P(k) \cdot rel(k)$$

$$P@k = \frac{|\text{relevant} \cap \text{top-k}|}{k} \qquad Recall@k = \frac{|\text{relevant} \cap \text{top-k}|}{|\text{relevant}|}$$

Trong đó $R_q$ là tổng số tài liệu liên quan cho query $q$, $rel(k)=1$ nếu tài liệu tại vị trí $k$ là relevant.

### Triển khai

In [19]:
def average_precision(retrieved, relevant):
    relevant_set = set(relevant)
    if not relevant_set: return 0.0
    score, hits = 0.0, 0
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant_set:
            hits  += 1
            score += hits / i
    return score / len(relevant_set)

def evaluate_model(results, query_rels, k=10):
    aps, ps, rs = [], [], []
    for qid, retrieved in results.items():
        if qid not in query_rels: continue
        relevant = set(query_rels[qid])
        if not relevant: continue
        top_k = retrieved[:k]
        hits  = sum(1 for d in top_k if d in relevant)
        ps.append(hits / k)
        rs.append(hits / len(relevant))
        aps.append(average_precision(retrieved, query_rels[qid]))
    return {
        'MAP':        round(float(np.mean(aps)), 4) if aps else 0.0,
        f'P@{k}':     round(float(np.mean(ps)),  4) if ps  else 0.0,
        f'Recall@{k}':round(float(np.mean(rs)),  4) if rs  else 0.0,
    }


#### Ví dụ chạy tay Đánh giá (Evaluation) cho Query 1 với k=5

**Giả sử Query 1 chạy qua mô hình VSM trả về top 5 kết quả:**
- **Relevant (Tập tài liệu thực sự liên quan)**: $R_{q_1} = 28$ tài liệu (bao gồm `51, 184, 12, ...`)
- **Retrieved (Top 5 tài liệu trả về)**: `[51, 184, 12, 359, 746]`
- **Intersection (Tài liệu truy xuất đúng trong top 5)**: `[51, 184, 12]` (3 tài liệu)

**1. Tính Precision@5 (P@5):**
Tỷ lệ tài liệu trả về đúng trên tổng số 5 tài liệu được hệ thống trả về.
$$P@5 = \frac{|\text{Relevant} \cap \text{Retrieved@5}|}{5} = \frac{3}{5} = 0.6000$$

**2. Tính Recall@5:**
Tỷ lệ tài liệu trả về đúng trên tổng số tài liệu thực sự liên quan của toàn hệ thống ($R_{q_1}=28$).
$$Recall@5 = \frac{|\text{Relevant} \cap \text{Retrieved@5}|}{|Relevant|} = \frac{3}{28} \approx 0.1071$$

**3. Tính Average Precision (AP):**
Xét từng vị trí $i$ từ 1 đến 5 để tính trung bình các Precision tại các vị trí dự đoán đúng (hits):
- Tại $i=1$, Doc `51` (Relevant ✅) $\rightarrow$ hits = 1, $P(1) = 1/1 = 1.000$
- Tại $i=2$, Doc `184` (Relevant ✅) $\rightarrow$ hits = 2, $P(2) = 2/2 = 1.000$
- Tại $i=3$, Doc `12` (Relevant ✅) $\rightarrow$ hits = 3, $P(3) = 3/3 = 1.000$
- Tại $i=4$, Doc `359` (Irrelevant ❌) $\rightarrow$ hits = 3, (Không cộng vào tổng vì $rel=0$)
- Tại $i=5$, Doc `746` (Irrelevant ❌) $\rightarrow$ hits = 3, (Không cộng vào tổng vì $rel=0$)

Tổng điểm = $1.000 + 1.000 + 1.000 = 3.000$
$$AP(q_1) = \frac{\text{Tổng điểm}}{|Relevant|} = \frac{3.000}{28} \approx 0.1071$$


In [20]:

# Chạy VSM
vsm_results = {}
for qid, qtext in queries.items():
    vsm_results[qid] = [d for d,_ in search_vsm(qtext, tfidf_matrix, vsm_index, vocab_index, k=20)]

# Chạy BM25
bm25_results = {}
for qid, qtext in queries.items():
    bm25_results[qid] = [d for d,_ in search_bm25(qtext, bm25_model, k=20)]

vsm_metrics  = evaluate_model(vsm_results,  query_rels, k=10)
bm25_metrics = evaluate_model(bm25_results, query_rels, k=10)

summary = pd.DataFrame([vsm_metrics, bm25_metrics], index=['VSM', 'BM25'])
summary.style.set_caption("Kết quả Đánh giá – Cranfield 1400 docs / 225 queries").set_table_styles([
    {'selector':'th','props':[('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
    {'selector':'td','props':[('padding','8px 18px'),('text-align','right'),('font-size','15px')]},
    {'selector':'tr:nth-child(even)','props':[('background-color','#f4f5ff')]},
])


,MAP,P@10,Recall@10
VSM,0.258800,0.234700,0.387900
BM25,0.280400,0.239600,0.401000


## 6. Phân tích Best / Worst Case (BM25)

Phân tích những câu truy vấn mà mô hình trả về kết quả **tốt nhất** và **tệ nhất** để hiểu điểm mạnh và điểm yếu của hệ thống.

In [21]:
# Tính AP cho từng query
ap_per_query = {qid: average_precision(bm25_results[qid], query_rels.get(qid,[]))
                for qid in bm25_results if qid in query_rels}

best_qid  = max(ap_per_query, key=ap_per_query.get)
worst_qid = min(ap_per_query, key=ap_per_query.get)

print(f"Best  query: Q{best_qid}  AP={ap_per_query[best_qid]:.4f}")
print(f"Worst query: Q{worst_qid}  AP={ap_per_query[worst_qid]:.4f}")


Best  query: Q119  AP=1.0000
Worst query: Q13  AP=0.0000


### Best Case – BM25

In [22]:
qid = best_qid
print(f"Query #{qid}: {queries[qid]}")
print(f"AP = {ap_per_query[qid]:.4f}  |  Relevant docs: {len(query_rels.get(qid,[]))}")

top_doc = bm25_results[qid][0]
q_tokens_best = process_document(queries[qid])
dl = bm25_model.doc_len[top_doc]

# Bảng phân tích score từng term
rows = []
for t in q_tokens_best:
    f    = bm25_model.doc_freqs[top_doc].get(t, 0)
    idf  = bm25_model.idf.get(t, 0)
    num  = f * (bm25_model.k1 + 1)
    den  = f + bm25_model.k1 * (1 - bm25_model.b + bm25_model.b * dl / bm25_model.avgdl)
    rows.append({'Term (query)': t, 'f(t,d)': f,
                 'IDF_BM25': round(idf,4),
                 'Numerator': round(num,4), 'Denominator': round(den,4),
                 'Term score': round(idf*(num/den) if den>0 and f>0 else 0, 4)})

sc_df = pd.DataFrame(rows)
total = sc_df['Term score'].sum()
print(f"\nDoc {top_doc} | |d|={dl} | Final BM25 score = {total:.4f}")
sc_df.style.set_caption(f"Score breakdown – Query {qid} × Doc {top_doc}").set_table_styles([
    {'selector':'th','props':[('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
    {'selector':'td','props':[('padding','5px 12px'),('text-align','right')]},
    {'selector':'td:first-child','props':[('text-align','left'),('color','#3b3f8c')]},
])


Query #119: what is the effect of initial axisymmetric deviations from circularity on the non linear (large-deflection) load-deflection response of cylinders under hydrostatic pressure .
AP = 1.0000  |  Relevant docs: 1

Doc 926 | |d|=80 | Final BM25 score = 30.0282


,Term (query),"f(t,d)",IDF_BM25,Numerator,Denominator,Term score
0,effect,0,0.952400,0.000000,1.804400,0.000000
1,initi,1,2.549000,3.000000,2.804400,2.726800
2,axisymmetr,0,3.265300,0.000000,1.804400,0.000000
3,deviat,0,3.967800,0.000000,1.804400,0.000000
4,circular,2,2.306900,6.000000,3.804400,3.638200
5,non,0,3.093900,0.000000,1.804400,0.000000
6,linear,0,2.148100,0.000000,1.804400,0.000000
7,larg,0,1.890700,0.000000,1.804400,0.000000
8,deflect,3,2.773300,9.000000,4.804400,5.195200
9,load,6,1.944100,18.000000,7.804400,4.483900


### Worst Case – Query tìm kiếm kém chính xác nhất

In [23]:
qid = worst_qid
print(f"Query #{qid}: {queries[qid]}")
print(f"AP = {ap_per_query[qid]:.4f}\n")

relevant_docs = query_rels.get(qid, [])
retrieved_top5 = bm25_results[qid][:5]

print(f"Relevant docs (first 10): {relevant_docs[:10]}")
print(f"Retrieved top-5          : {retrieved_top5}")
overlap = set(retrieved_top5) & set(relevant_docs)
print(f"Overlap in top-5         : {overlap if overlap else 'Không có (False Positive hoàn toàn)'}")


Query #13: what is the basic mechanism of the transonic aileron buzz .
AP = 0.0000

Relevant docs (first 10): [64, 265, 65, 311]
Retrieved top-5          : [496, 903, 520, 643, 199]
Overlap in top-5         : Không có (False Positive hoàn toàn)
